In [1]:
# If needed, install extra libs (run once in the environment)
# !pip install -U transformers datasets textstat wordfreq accelerate sentencepiece

from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import numpy as np
import textstat
from wordfreq import zipf_frequency
import re
from collections import Counter
from typing import Dict, List
import evaluate


C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  from scipy.sparse import csr_matrix, issparse


In [2]:
from datasets import DatasetDict

simpl_ds = load_dataset("eilamc14/wikilarge-clean")

# Take a subset for speed
max_train = 100
max_val = 10

train_ds = simpl_ds["train"].shuffle(seed=43).select(range(min(max_train, len(simpl_ds["train"]))))
val_ds = simpl_ds["validation"].shuffle(seed=43).select(range(min(max_val, len(simpl_ds["validation"]))))

simpl_small = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": simpl_ds["test"]  # optional
})
print(simpl_small['train'][0])


{'source': 'Those who have the formal power to create legislation are known as legislators ; a judicial branch of government will have the formal power to interpret legislation ( see statutory interpretation ) ; the executive branch of government can act only within the powers and limits set by the law .', 'target': 'Those who have the formal power to create legislation are known as legislators , the judicial branch of government may have the formal power to interpret legislation .'}


In [3]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [5]:
def readability_metrics(text: str) -> Dict[str, float]:
    """Compute basic readability metrics for a text."""
    return {
        "fk_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text),
        "smog_index": textstat.smog_index(text),
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
    }


def tokenize_words(text: str) -> List[str]:
    return re.findall(r"\b[a-zA-Z][a-zA-Z\-]+\b", text.lower())


def is_common_word(word: str, threshold: float = 4.5) -> bool:
    """
    Uses Zipf frequency (0–7, higher = more common).
    ~4–5: common everyday English; <4: rare/technical.
    """
    return zipf_frequency(word, "en") >= threshold


def jargon_ratio(text: str, threshold: float = 4.5, min_len: int = 4) -> float:
    """
    Approximate jargon density using wordfreq.
    Returns % of tokens considered jargon.
    """
    words = tokenize_words(text)
    if not words:
        return 0.0
    jargon_words = [
        w for w in words
        if len(w) >= min_len and not is_common_word(w, threshold)
    ]
    return 100.0 * len(jargon_words) / len(words)


In [6]:
max_input_length = 256
max_target_length = 128

def preprocess_function(batch):
    inputs = []
    for src in batch["source"]:
        # Instruction-style prefix; you can tune wording
        inputs.append(f"simplify for better reading and understanding: {src}")

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

tokenized_simpl_ds = simpl_small.map(
    preprocess_function,
    batched=True,
    remove_columns=simpl_small["train"].column_names,
)
tokenized_simpl_ds


Map:   0%|          | 0/100 [00:00<?, ? examples/s]C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 121/121 [00:00<00:00, 2839.78 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 121
    })
})

In [7]:
# ROUGE for overlap with reference simplifications
rouge = evaluate.load("rouge")


def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 (ignore index) with tokenizer.pad_token_id
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # For labels we need to remove ignored index (-100)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # ROUGE
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    # Readability & jargon – average over batch
    fk_grades = []
    fre_scores = []
    jargon_ratios = []

    for text in decoded_preds:
        rm = readability_metrics(text)
        fk_grades.append(rm["fk_grade"])
        fre_scores.append(rm["flesch_reading_ease"])
        jargon_ratios.append(jargon_ratio(text))

    metrics = {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "fk_grade_pred": float(np.nanmean(fk_grades)),
        "fre_pred": float(np.nanmean(fre_scores)),
        "jargon_pct_pred": float(np.nanmean(jargon_ratios)),
    }

    # Higher Flesch reading ease is better, lower FK grade & jargon_pct ideally
    return metrics


In [8]:
output_dir = "./t5-simplification-wikilarge"

batch_size = 8  # adjust to your GPU
num_train_epochs = 3

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=100,
    report_to=["none"],  # or ["tensorboard"]
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",  # you could also define a custom composite metric offline
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_simpl_ds["train"],
    eval_dataset=tokenized_simpl_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save final model + tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)


C:\Users\Ilinca\AppData\Local\Temp\ipykernel_7340\3859798444.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Fk Grade Pred,Fre Pred,Jargon Pct Pred
1,No log,11.938905,0.337750,0.195984,0.334015,10.456106,40.224110,27.924550
2,No log,8.990066,0.319318,0.174559,0.318788,10.499796,39.732769,27.021772
3,No log,7.024199,0.308402,0.177279,0.308283,8.832583,28.398668,28.968201


C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


('./t5-simplification-wikilarge\\tokenizer_config.json',
 './t5-simplification-wikilarge\\special_tokens_map.json',
 './t5-simplification-wikilarge\\spiece.model',
 './t5-simplification-wikilarge\\added_tokens.json',
 './t5-simplification-wikilarge\\tokenizer.json')

In [ ]:
# Load your clinical dataset (already preprocessed in Part 1)
# Make sure this path matches where you saved filtered_ds / work_ds
clinical_ds = load_from_disk("data/filtered_ds")

# Column from Step 1: technical summary to simplify
TECH_SUMMARY_COL = "summary_step1"  # change if needed, e.g. "generated_summary"

# Fallback: if you don't have summary_step1 yet, you *could* temporarily use brief_summary
# TECH_SUMMARY_COL = "brief_summary"

# Load the generic simplifier trained on WikiLarge (Stage A)
base_model_dir = "./t5-simplification-wikilarge"
tokenizer = AutoTokenizer.from_pretrained(base_model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(base_model_dir)
model.eval()

In [ ]:
def add_simplify_input(example):
    text = example[TECH_SUMMARY_COL]
    if text is None:
        text = ""
    example["simplify_input"] = f"simplify to patient-friendly english: {text}"
    example["simplify_source_text"] = text
    return example

clinical_ds = clinical_ds.map(add_simplify_input)

In [ ]:
import torch


def generate_and_score(batch):
    texts = batch["simplify_input"]
    source_texts = batch["simplify_source_text"]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            num_beams=4,
        )

    decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    fk_in, fre_in, jar_in = [], [], []
    gf_in, smog_in = [], []

    fk_out, fre_out, jar_out = [], [], []
    gf_out, smog_out = [], []

    for src, out in zip(source_texts, decoded):
        rm_in = readability_metrics(src)
        rm_out = readability_metrics(out)

        fk_in.append(rm_in["fk_grade"])
        gf_in.append(rm_in["gunning_fog"])
        smog_in.append(rm_in["smog_index"])
        fre_in.append(rm_in["flesch_reading_ease"])
        jar_in.append(jargon_ratio(src))

        fk_out.append(rm_out["fk_grade"])
        gf_out.append(rm_out["gunning_fog"])
        smog_out.append(rm_out["smog_index"])
        fre_out.append(rm_out["flesch_reading_ease"])
        jar_out.append(jargon_ratio(out))

    batch["pseudo_simple"] = decoded

    batch["fk_in"] = fk_in
    batch["gunning_fog_in"] = gf_in
    batch["smog_in"] = smog_in
    batch["fre_in"] = fre_in
    batch["jargon_in"] = jar_in

    batch["fk_out"] = fk_out
    batch["gunning_fog_out"] = gf_out
    batch["smog_out"] = smog_out
    batch["fre_out"] = fre_out
    batch["jargon_out"] = jar_out

    return batch

clinical_with_pseudo = clinical_ds.map(
    generate_and_score,
    batched=True,
    batch_size=8,
)

In [ ]:
def is_good_example(example,
                    min_fk_improvement: float = 1.0,
                    min_fre_improvement: float = 5.0,
                    min_jargon_drop: float = 5.0) -> bool:
    # Note: textstat sometimes returns -inf / nan
    fk_in = example["fk_in"]
    fk_out = example["fk_out"]
    fre_in = example["fre_in"]
    fre_out = example["fre_out"]
    jar_in = example["jargon_in"]
    jar_out = example["jargon_out"]

    # Avoid NaNs
    try:
        fk_improv = fk_in - fk_out
        fre_improv = fre_out - fre_in
        jargon_drop = jar_in - jar_out
    except TypeError:
        return False

    # Require all three improvements
    if np.isnan(fk_improv) or np.isnan(fre_improv) or np.isnan(jargon_drop):
        return False

    return (
        fk_improv >= min_fk_improvement and
        fre_improv >= min_fre_improvement and
        jargon_drop >= min_jargon_drop
    )

pseudo_train_ds = clinical_with_pseudo.filter(is_good_example)
print(len(pseudo_train_ds))


In [ ]:
from datasets import DatasetDict

# If you want, split into train/val
pseudo_train_val = pseudo_train_ds.train_test_split(test_size=0.1, seed=42)
train_ds = pseudo_train_val["train"]
val_ds = pseudo_train_val["test"]

max_input_length = 256
max_target_length = 256  # summaries may be a bit longer

def preprocess_pseudo(batch):
    inputs = [
        f"simplify to patient-friendly english: {txt}"
        for txt in batch["simplify_source_text"]
    ]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["pseudo_simple"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

tokenized_train = train_ds.map(
    preprocess_pseudo,
    batched=True,
    remove_columns=train_ds.column_names,
)

tokenized_val = val_ds.map(
    preprocess_pseudo,
    batched=True,
    remove_columns=val_ds.column_names,
)


In [ ]:
output_dir = "./t5-simplification-clinical-pseudo"

batch_size = 8
num_train_epochs = 2  # often 1–3 is enough for adaptation

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,  # a bit lower for continued fine-tuning
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=100,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="fk_grade_pred",  # or "rougeL" or custom
    greater_is_better=False,  # lower FK grade is better
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
